In [0]:
# Databricks notebook source

from pyspark.sql import functions as F

bronze_table = "fintech_lakehouse.bronze.orders_raw"
silver_table = "fintech_lakehouse.silver.orders_current"
gold_table = "fintech_lakehouse.gold.daily_order_kpis"

test_results = []


# Test 1: Bronze table must contain data

bronze_count = spark.table(bronze_table).count()

test_results.append({
    "test": "Bronze table is not empty",
    "result": "PASS" if bronze_count > 0 else "FAIL",
    "value": bronze_count
})

assert bronze_count > 0, "Bronze table is empty"



# Test 2: Removing null to clean the Silver, and order IDs must not be null

null_order_ids = (
    spark.table(silver_table)
    .filter(F.col("order_id").isNull())
    .count()
)

test_results.append({
    "test": "No null order IDs",
    "result": "PASS" if null_order_ids == 0 else "FAIL",
    "value": null_order_ids
})

assert null_order_ids == 0, \
    f"Found {null_order_ids} null order IDs"



# Test 3: Silver must have one current row per order

duplicate_orders = (
    spark.table(silver_table)
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

test_results.append({
    "test": "No duplicate current orders",
    "result": "PASS" if duplicate_orders == 0 else "FAIL",
    "value": duplicate_orders
})

assert duplicate_orders == 0, \
    f"Found {duplicate_orders} duplicate orders"


# Test 4: Order amounts cannot be negative

negative_amounts = (
    spark.table(silver_table)
    .filter(F.col("amount") < 0)
    .count()
)

test_results.append({
    "test": "No negative order amounts",
    "result": "PASS" if negative_amounts == 0 else "FAIL",
    "value": negative_amounts
})

assert negative_amounts == 0, \
    f"Found {negative_amounts} negative amounts"


# Test 5: Only accepted statuses are allowed

valid_statuses = ["CREATED", "PAID", "SHIPPED", "CANCELLED"]

invalid_statuses = (
    spark.table(silver_table)
    .filter(
        F.col("status").isNull() |
        ~F.col("status").isin(valid_statuses)
    )
    .count()
)

test_results.append({
    "test": "Only valid statuses",
    "result": "PASS" if invalid_statuses == 0 else "FAIL",
    "value": invalid_statuses
})

assert invalid_statuses == 0, \
    f"Found {invalid_statuses} invalid statuses"


# Test 6: Gold table must contain data

gold_count = spark.table(gold_table).count()

test_results.append({
    "test": "Gold KPI table is not empty",
    "result": "PASS" if gold_count > 0 else "FAIL",
    "value": gold_count
})

assert gold_count > 0, "Gold KPI table is empty"

#testing results

results_df = spark.createDataFrame(test_results)

display(results_df)

print("All data-quality checks passed successfully.")